In [2]:
#!/usr/bin/env python3

import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy import stats
from scipy.optimize import curve_fit
import os, json, warnings
warnings.filterwarnings("ignore")

# ── reproducibility ──────────────────────────────────────────────
np.random.seed(42)

# ── output directory ─────────────────────────────────────────────
OUT = os.path.join(os.getcwd(), "figures") # Changed from os.path.dirname(os.path.abspath(__file__))
os.makedirs(OUT, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════
# 1.  BUILD THE POWER GRID NETWORK
# ═══════════════════════════════════════════════════════════════════

def build_grid(N=200, m=3, topology="scale_free"):
    """
    Create a power-grid–like network.

    Parameters
    ----------
    N : int
        Number of nodes (substations).
    m : int
        Edges to attach for each new node (Barabási-Albert model).
    topology : str
        'scale_free'  – Barabási-Albert  (realistic for power grids)
        'small_world' – Watts-Strogatz
        'random'      – Erdős-Rényi

    Returns
    -------
    G : networkx.Graph  with node attributes:
        'load'     – current load (starts at 0)
        'capacity' – maximum load before failure
        'failed'   – boolean failure flag
    """
    if topology == "scale_free":
        G = nx.barabasi_albert_graph(N, m, seed=42)
    elif topology == "small_world":
        G = nx.watts_strogatz_graph(N, 2 * m, 0.3, seed=42)
    elif topology == "random":
        p = 2 * m / N
        G = nx.erdos_renyi_graph(N, p, seed=42)
    else:
        raise ValueError(f"Unknown topology: {topology}")

    for node in G.nodes():
        deg = G.degree(node)
        # capacity proportional to degree (hubs can handle more)
        G.nodes[node]['capacity'] = max(deg * 1.0, 1.0)
        G.nodes[node]['load'] = 0.0
        G.nodes[node]['failed'] = False

    return G


# ═══════════════════════════════════════════════════════════════════
# 2.  SAND-PILE / CASCADE DYNAMICS
# ═══════════════════════════════════════════════════════════════════

def add_noise(G, magnitude=1.0):
    """
    Inject a random load perturbation ('noise') at a random alive node.
    Returns the perturbed node index.
    """
    alive = [n for n in G.nodes() if not G.nodes[n]['failed']]
    if not alive:
        return None
    node = np.random.choice(alive)
    G.nodes[node]['load'] += magnitude
    return node


def cascade_step(G, redistribution_fraction=0.8):
    """
    One relaxation sweep: every overloaded node fails and redistributes
    its load to alive neighbours.

    Returns
    -------
    newly_failed : list of nodes that failed in this sweep
    """
    newly_failed = []
    for n in list(G.nodes()):
        nd = G.nodes[n]
        if nd['failed']:
            continue
        if nd['load'] > nd['capacity']:
            nd['failed'] = True
            newly_failed.append(n)
            # redistribute load to alive neighbours
            alive_nbrs = [nb for nb in G.neighbors(n)
                          if not G.nodes[nb]['failed']]
            if alive_nbrs:
                share = (nd['load'] * redistribution_fraction) / len(alive_nbrs)
                for nb in alive_nbrs:
                    G.nodes[nb]['load'] += share
            nd['load'] = 0.0
    return newly_failed


def run_avalanche(G, redistribution_fraction=0.8):
    """
    After a perturbation, propagate cascades until no more nodes fail.

    Returns
    -------
    avalanche_size : int   – total number of nodes that failed
    failed_nodes   : list  – list of failed node indices
    steps          : int   – number of cascade steps
    """
    total_failed = []
    steps = 0
    while True:
        nf = cascade_step(G, redistribution_fraction)
        if not nf:
            break
        total_failed.extend(nf)
        steps += 1
    return len(total_failed), total_failed, steps


def reset_system(G):
    """Reset all nodes to alive with zero load."""
    for n in G.nodes():
        G.nodes[n]['load'] = 0.0
        G.nodes[n]['failed'] = False


# ═══════════════════════════════════════════════════════════════════
# 3.  MAIN SIMULATION LOOP – DRIVE TO SOC
# ═══════════════════════════════════════════════════════════════════

def simulate(N=200, m=3, topology="scale_free",
             T=20000, noise_mag=0.25,
             redistribution_fraction=0.8):
    """
    Drive-dissipation simulation.

    Each time step:
      1. Add random noise (drive).
      2. Propagate any cascade (relaxation).
      3. Record avalanche size.
      4. If system is significantly damaged (>30 % failed), partially
         repair it (slow recovery) to keep the system near criticality.

    Returns
    -------
    results : dict with keys
        'avalanche_sizes', 'avalanche_steps', 'time_series',
        'graph', 'connectivity_history'
    """
    G = build_grid(N, m, topology)
    avalanche_sizes = []
    avalanche_steps = []
    connectivity_history = []  # track largest connected component fraction

    for t in range(T):
        # — drive —
        add_noise(G, magnitude=noise_mag)

        # — relaxation —
        av_size, failed_list, steps = run_avalanche(G, redistribution_fraction)
        avalanche_sizes.append(av_size)
        avalanche_steps.append(steps)

        # — measure connectivity —
        alive_subgraph = G.subgraph(
            [n for n in G.nodes() if not G.nodes[n]['failed']]
        )
        if alive_subgraph.number_of_nodes() > 0:
            largest_cc = max(nx.connected_components(alive_subgraph), key=len)
            conn = len(largest_cc) / N
        else:
            conn = 0.0
        connectivity_history.append(conn)

        # — slow repair (dissipation) to sustain criticality —
        n_failed = sum(1 for n in G.nodes() if G.nodes[n]['failed'])
        if n_failed > 0.30 * N:
            # repair some random failed nodes
            failed_nodes = [n for n in G.nodes() if G.nodes[n]['failed']]
            n_repair = max(1, int(0.1 * len(failed_nodes)))
            to_repair = np.random.choice(failed_nodes, size=n_repair,
                                         replace=False)
            for n in to_repair:
                G.nodes[n]['failed'] = False
                G.nodes[n]['load'] = 0.0

    return {
        'avalanche_sizes': avalanche_sizes,
        'avalanche_steps': avalanche_steps,
        'connectivity_history': connectivity_history,
        'graph': G,
        'N': N, 'm': m, 'topology': topology,
        'T': T, 'noise_mag': noise_mag,
    }


# ═══════════════════════════════════════════════════════════════════
# 4.  ANALYSIS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def power_law(x, a, alpha):
    """Power-law function: f(x) = a * x^(-alpha)."""
    return a * np.power(x, -alpha)


def fit_power_law(sizes):
    """
    Fit a power-law to the avalanche-size frequency distribution.
    Returns (alpha, a, x_data, y_data, y_fit).
    """
    sizes = [s for s in sizes if s > 0]
    if not sizes:
        return None
    counter = Counter(sizes)
    x = np.array(sorted(counter.keys()), dtype=float)
    y = np.array([counter[int(xi)] for xi in x], dtype=float)
    # normalise to probability
    y = y / y.sum()

    try:
        popt, _ = curve_fit(power_law, x, y, p0=[1.0, 1.5],
                            maxfev=10000)
        a_fit, alpha_fit = popt
        y_fit = power_law(x, a_fit, alpha_fit)
        return alpha_fit, a_fit, x, y, y_fit
    except Exception:
        return None


def compute_ccdf(sizes):
    """Complementary cumulative distribution function."""
    sizes = sorted([s for s in sizes if s > 0])
    n = len(sizes)
    unique = sorted(set(sizes))
    ccdf_x, ccdf_y = [], []
    for val in unique:
        ccdf_x.append(val)
        ccdf_y.append(sum(1 for s in sizes if s >= val) / n)
    return np.array(ccdf_x), np.array(ccdf_y)


# ═══════════════════════════════════════════════════════════════════
# 5.  PLOTTING
# ═══════════════════════════════════════════════════════════════════

COLORS = {
    'primary': '#1a5276',
    'secondary': '#c0392b',
    'tertiary': '#27ae60',
    'accent': '#f39c12',
    'bg': '#fafafa',
}


def plot_network(G, title="Power Grid Network", filename="network.png"):
    """Visualise the network coloured by load/capacity ratio."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    pos = nx.spring_layout(G, seed=42, k=0.5)

    alive = [n for n in G.nodes() if not G.nodes[n]['failed']]
    failed = [n for n in G.nodes() if G.nodes[n]['failed']]

    # colour alive nodes by load fraction
    load_frac = []
    for n in alive:
        cap = G.nodes[n]['capacity']
        ld = G.nodes[n]['load']
        load_frac.append(ld / cap if cap > 0 else 0)

    nx.draw_networkx_edges(G, pos, alpha=0.15, ax=ax)
    if alive:
        nc = nx.draw_networkx_nodes(G, pos, nodelist=alive,
                                    node_color=load_frac,
                                    cmap=plt.cm.YlOrRd,
                                    node_size=40, ax=ax,
                                    vmin=0, vmax=1.2)
        plt.colorbar(nc, ax=ax, label="Load / Capacity", shrink=0.7)
    if failed:
        nx.draw_networkx_nodes(G, pos, nodelist=failed,
                               node_color='black', node_size=20,
                               ax=ax, alpha=0.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


def plot_avalanche_timeseries(sizes, filename="avalanche_timeseries.png"):
    """Time series of avalanche sizes."""
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.bar(range(len(sizes)), sizes, width=1.0, color=COLORS['primary'],
           alpha=0.7, linewidth=0)
    ax.set_xlabel("Time step", fontsize=11)
    ax.set_ylabel("Avalanche size", fontsize=11)
    ax.set_title("Cascade (Avalanche) Sizes Over Time", fontsize=13,
                 fontweight='bold')
    ax.set_xlim(0, len(sizes))
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


def plot_frequency_distribution(sizes, filename="freq_distribution.png"):
    """Log-log frequency distribution with power-law fit."""
    result = fit_power_law(sizes)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- PDF (log-log) ---
    ax = axes[0]
    if result is not None:
        alpha, a_fit, x, y, y_fit = result
        ax.scatter(x, y, s=20, color=COLORS['primary'], zorder=3,
                   label="Data")
        ax.plot(x, y_fit, '-', color=COLORS['secondary'], lw=2,
                label=f"Fit: α = {alpha:.2f}")
        ax.legend(fontsize=10)
    else:
        counter = Counter([s for s in sizes if s > 0])
        x = sorted(counter.keys())
        y = [counter[xi] for xi in x]
        ax.scatter(x, y, s=20, color=COLORS['primary'])
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Avalanche size s", fontsize=11)
    ax.set_ylabel("P(s)", fontsize=11)
    ax.set_title("Avalanche Size Distribution (PDF)", fontsize=13,
                 fontweight='bold')

    # --- CCDF ---
    ax2 = axes[1]
    ccdf_x, ccdf_y = compute_ccdf(sizes)
    ax2.scatter(ccdf_x, ccdf_y, s=15, color=COLORS['tertiary'], zorder=3)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel("Avalanche size s", fontsize=11)
    ax2.set_ylabel("P(S ≥ s)", fontsize=11)
    ax2.set_title("Complementary CDF", fontsize=13, fontweight='bold')

    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")
    return result


def plot_connectivity(conn_hist, filename="connectivity.png"):
    """Largest connected component fraction over time."""
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(conn_hist, color=COLORS['tertiary'], lw=0.6, alpha=0.8)
    ax.axhline(y=np.mean(conn_hist), color=COLORS['secondary'],
               ls='--', lw=1.5, label=f"Mean = {np.mean(conn_hist):.3f}")
    ax.fill_between(range(len(conn_hist)), conn_hist,
                    alpha=0.15, color=COLORS['tertiary'])
    ax.set_xlabel("Time step", fontsize=11)
    ax.set_ylabel("Largest CC / N", fontsize=11)
    ax.set_title("Network Connectivity Over Time", fontsize=13,
                 fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


def plot_connectivity_vs_avalanche(sizes, conn_hist,
                                    filename="conn_vs_avalanche.png"):
    """Scatter: connectivity drop vs avalanche size."""
    # compute connectivity drops
    drops = [0] + [max(0, conn_hist[i-1] - conn_hist[i])
                   for i in range(1, len(conn_hist))]
    nonzero = [(sizes[i], drops[i]) for i in range(len(sizes))
               if sizes[i] > 0 and drops[i] > 0]
    if not nonzero:
        print("  ⚠ No data for connectivity-vs-avalanche plot")
        return

    av_s, dr = zip(*nonzero)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(av_s, dr, s=12, alpha=0.4, color=COLORS['primary'])
    ax.set_xlabel("Avalanche size", fontsize=11)
    ax.set_ylabel("Connectivity drop", fontsize=11)
    ax.set_title("Connectivity Drop vs Avalanche Size", fontsize=13,
                 fontweight='bold')
    ax.set_xscale('log')
    ax.set_yscale('log')
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


def plot_topology_comparison(results_dict, filename="topology_comparison.png"):
    """Compare avalanche distributions across topologies."""
    fig, ax = plt.subplots(figsize=(7, 5))
    colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['tertiary']]
    for i, (topo, res) in enumerate(results_dict.items()):
        ccdf_x, ccdf_y = compute_ccdf(res['avalanche_sizes'])
        ax.scatter(ccdf_x, ccdf_y, s=15, alpha=0.6,
                   color=colors_list[i % 3], label=topo)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Avalanche size s", fontsize=11)
    ax.set_ylabel("P(S ≥ s)", fontsize=11)
    ax.set_title("Avalanche CCDF by Network Topology", fontsize=13,
                 fontweight='bold')
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


def plot_degree_distribution(G, filename="degree_distribution.png"):
    """Degree distribution of the network."""
    degrees = [G.degree(n) for n in G.nodes()]
    counter = Counter(degrees)
    x = sorted(counter.keys())
    y = [counter[k] for k in x]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.bar(x, y, color=COLORS['accent'], alpha=0.8, edgecolor='white')
    ax.set_xlabel("Degree k", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title("Node Degree Distribution", fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, filename), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved {filename}")


# ═══════════════════════════════════════════════════════════════════
# 6.  MAIN
# ═══════════════════════════════════════════════════════════════════

def main():
    print("=" * 65)
    print("  POWER GRID FAILURE CASCADES – SOC SIMULATION")
    print("=" * 65)

    # ── Primary simulation: Scale-free topology ──────────────────
    print("\n[1/4]  Running primary simulation (scale-free, N=200, T=20000) ...")
    res_sf = simulate(N=200, m=3, topology="scale_free",
                      T=20000, noise_mag=0.25,
                      redistribution_fraction=0.8)
    G = res_sf['graph']
    sizes = res_sf['avalanche_sizes']
    conn = res_sf['connectivity_history']

    # ── Plots ────────────────────────────────────────────────────
    print("\n[2/4]  Generating plots ...")
    plot_network(G, title="Power Grid – Scale-Free Network (N=200)")
    plot_degree_distribution(G)
    plot_avalanche_timeseries(sizes)
    pw = plot_frequency_distribution(sizes)
    plot_connectivity(conn)
    plot_connectivity_vs_avalanche(sizes, conn)

    # ── Topology comparison ──────────────────────────────────────
    print("\n[3/4]  Running topology comparison ...")
    res_sw = simulate(N=200, m=3, topology="small_world",
                      T=20000, noise_mag=0.25,
                      redistribution_fraction=0.8)
    res_rn = simulate(N=200, m=3, topology="random",
                      T=20000, noise_mag=0.25,
                      redistribution_fraction=0.8)
    plot_topology_comparison({
        "Scale-Free": res_sf,
        "Small-World": res_sw,
        "Random": res_rn,
    })

    # ── Summary statistics ───────────────────────────────────────
    print("\n[4/4]  Summary Statistics")
    print("-" * 45)
    nonzero = [s for s in sizes if s > 0]
    print(f"  Total time steps       : {len(sizes)}")
    print(f"  Non-zero avalanches    : {len(nonzero)}")
    print(f"  Max avalanche size     : {max(sizes)}")
    print(f"  Mean avalanche (>0)    : {np.mean(nonzero):.2f}")
    print(f"  Median avalanche (>0)  : {np.median(nonzero):.1f}")
    if pw is not None:
        print(f"  Power-law exponent α   : {pw[0]:.3f}")
    print(f"  Mean connectivity      : {np.mean(conn):.4f}")
    print(f"  Min  connectivity      : {np.min(conn):.4f}")
    print("-" * 45)

    # save stats to JSON for the report
    stats_out = {
        'total_steps': len(sizes),
        'nonzero_avalanches': len(nonzero),
        'max_avalanche': int(max(sizes)),
        'mean_avalanche': float(np.mean(nonzero)),
        'median_avalanche': float(np.median(nonzero)),
        'alpha': float(pw[0]) if pw else None,
        'mean_connectivity': float(np.mean(conn)),
        'min_connectivity': float(np.min(conn)),
    }
    with open(os.path.join(OUT, "stats.json"), 'w') as f:
        json.dump(stats_out, f, indent=2)

    print(f"\n✓ All figures saved to {OUT}/")
    print("=" * 65)


if __name__ == "__main__":
    main()


  POWER GRID FAILURE CASCADES – SOC SIMULATION

[1/4]  Running primary simulation (scale-free, N=200, T=20000) ...

[2/4]  Generating plots ...
  ✓ Saved network.png
  ✓ Saved degree_distribution.png
  ✓ Saved avalanche_timeseries.png
  ✓ Saved freq_distribution.png
  ✓ Saved connectivity.png
  ✓ Saved conn_vs_avalanche.png

[3/4]  Running topology comparison ...
  ✓ Saved topology_comparison.png

[4/4]  Summary Statistics
---------------------------------------------
  Total time steps       : 20000
  Non-zero avalanches    : 633
  Max avalanche size     : 139
  Mean avalanche (>0)    : 3.50
  Median avalanche (>0)  : 1.0
  Power-law exponent α   : 2.314
  Mean connectivity      : 0.7347
  Min  connectivity      : 0.0050
---------------------------------------------

✓ All figures saved to /content/figures/
